# Page 7 — "Frequency Voting": run the chain many times and keep what recurs

This notebook samples many chains of thought from Qwen2.5-3B on each of 100 hard maths problems
(MATH-500, levels 4 and 5, page 6's set) and keeps the answer most chains agree on. It measures
two things:

- **the pile**: N chains, each free to run to its own end, and how the vote's accuracy grows with N.
- **the split**: N chains sharing one budget of `B` tokens, `B/N` each, against one chain of `B`.

Chains are sampled at several temperatures. At temperature 0 every chain is the same, so the vote
changes nothing.

**What it writes**, in `out/page_07/`:

| File | What it holds |
|---|---|
| `traces.jsonl` | every chain, token by token (page 8's input) |
| `results.json` | every number this notebook computes, and the claims the page checks |
| `manifest.json` | how, where and with what the run was made |

`page_07_site_data.py` turns these into the `derived.json` the website draws. Shared helpers live
in `harness.py`. Both ship beside this notebook.

## 0. How to run this

| Mode | Model | Items | Pile | Temperatures | Hardware | Time |
|---|---|---|---|---|---|---|
| **`REPLAY`** *(default)* | none loaded | 100 | 16 chains | 5 | **no GPU** | **~2 min** |
| `QUICK` | Qwen2.5-3B | 20 | 8 chains | 2 | ~8 GB GPU | ~20 min |
| `FULL` | Qwen2.5-3B | 100 | 16 chains | 5 | 16 GB | ~2 h batched |

`QUICK` and `FULL` generate new chains. On different hardware, a different batch size or a
different `transformers` version they will not match ours chain for chain: expect the same
shape and ranking, not the same numbers. `REPLAY` is the only mode that reproduces the
published figures exactly.

The N axis is drawn from one pile: accuracy at N = 4 is the average over random 4-subsets of the
same 16 chains. Section 5 prints how long a generating run will take on this machine.

Choose the mode in the next cell. Install the dependencies with `pip install -r requirements.txt`.

# Part 1: setup and the sweep

Everything here produces `traces.jsonl`. In `REPLAY` it loads no model and takes about a minute.

In [ ]:
# ---------------------------------------------------------------------------
# The knobs. Everything else is derived from these.
# ---------------------------------------------------------------------------
RUN_MODE = "REPLAY"       # "REPLAY" (no GPU), "QUICK" (small pile), "FULL"
SEED = 123                # selects items, and seeds every batch of chains
DATASET = "math500_l45"   # MATH-500 levels 4 and 5, page 6's band

PRIMARY = "Qwen/Qwen2.5-3B-Instruct"        # page 6's primary, so the pages compare

PRESETS = {
    "QUICK":  dict(n_items=20, pool_n=8, temperatures=[0.0, 0.9],
                   diversity_n=4, diversity_items=10, split_n=[1, 4]),
    "FULL":   dict(n_items=100, pool_n=16, temperatures=[0.0, 0.3, 0.6, 0.9, 1.2],
                   diversity_n=8, diversity_items=30, split_n=[1, 4, 16]),
}
# REPLAY reads the FULL run, so it takes FULL's settings.
PRESETS["REPLAY"] = PRESETS["FULL"]
CFG = PRESETS[RUN_MODE]

# ---- the pile: N chains, each running to its natural end -------------------
# Page 6's top cap; most chains finish well before it.
CHAIN_CAP = 2048
POOL_N = CFG["pool_n"]
# Drawn as subsets of the pile, ending at the whole pile.
N_AXIS = sorted({n for n in (1, 2, 4, 8, 16, 32) if n <= POOL_N} | {POOL_N})
N_RESAMPLES = 64          # random subsets per (item, N)

# ---- the split: N chains sharing one budget --------------------------------
B_TOTAL = 2048            # page 6's top cap
SPLIT_N = CFG["split_n"]  # B/N tokens each

# ---- diversity -------------------------------------------------------------
TEMPERATURES = sorted(CFG["temperatures"])
TOP_P = 0.95              # held constant so temperature is the only diversity knob
# The diversity family runs on fewer problems than the pile.
DIVERSITY_N = CFG["diversity_n"]
DIVERSITY_ITEMS = CFG["diversity_items"]

# ---- generation ------------------------------------------------------------
GROUP = 8                 # chains per generate call, and the unit of reproduction
ANSWER_ALLOWANCE = 32     # tokens a forced answer may use

print(f"mode={RUN_MODE}  dataset={DATASET}  items={CFG['n_items']}")
print(f"the pile   {POOL_N} chains at a {CHAIN_CAP}-token cap  ->  N axis {N_AXIS} "
      f"by subsampling")
print(f"the split  B={B_TOTAL} shared by N in {SPLIT_N}  ->  chain budgets "
      f"{[B_TOTAL // n for n in SPLIT_N]}")
print(f"diversity  T in {TEMPERATURES} at N={DIVERSITY_N} on {DIVERSITY_ITEMS} items")
print(f"top_p={TOP_P}   group={GROUP}   answer allowance={ANSWER_ALLOWANCE}")

In [ ]:
import json
import math
import random
import textwrap
from collections import Counter
from pathlib import Path

import torch

# Shared plumbing, shipped beside this notebook.
import harness

from reasoning_from_scratch.ch02 import get_device
from reasoning_from_scratch.ch03 import (
    load_math500_test,
    render_prompt,
    extract_final_candidate,
    grade_answer,
)

OUT_DIR = Path("out/page_07")
OUT_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR = Path("data")
DATA_DIR.mkdir(parents=True, exist_ok=True)

TRACES_PATH = OUT_DIR / "traces.jsonl"
RESULTS_PATH = OUT_DIR / "results.json"
MANIFEST_PATH = OUT_DIR / "manifest.json"
PAGE_06_DIR = Path("out/page_06")

mean, median = harness.mean, harness.median


print("torch", torch.__version__)
print("harness", harness.fingerprint()["version"], harness.fingerprint()["sha256"][:12])

## 1. Grading

Chapter 3's grader from *Reasoning from Scratch*, unchanged
([`ch03.py`](https://github.com/rasbt/reasoning-from-scratch/blob/main/reasoning_from_scratch/ch03.py): `extract_final_candidate`, `grade_answer`). The last two
checks are chains cut off at their cap and forced to write an answer.

In [ ]:
CHECKS = [
    # (raw completion, ground truth, expected)
    (r"The answer is \boxed{9}.",                              "9",     True),
    (r"27",                                                     "9",     False),
    (r"\boxed{\frac{1}{2}}",                                   "0.5",   True),   # LaTeX vs decimal
    (r"\boxed{1,024}",                                         "1024",  True),   # thousands separator
    # cut at its cap, then forced to commit:
    ("Common denominator is 12, so 3/4 = 9/12 and 2/3 = 8/12. Now I need to subtr"
     "\n\nFinal answer: " + r"\boxed{\frac{17}{12}}",          r"\frac{17}{12}", True),
    # forced, but wrong:
    ("...\n\nFinal answer: " + r"\boxed{\frac{5}{12}}",        r"\frac{17}{12}", False),
]
harness.check_grader(CHECKS, extract_final_candidate, grade_answer)


## 2. The problems

Page 6's 100 problems: the same levels and the same seed, checked against page 6's manifest
when that run is on disk.

In [ ]:
math500 = load_math500_test(local_path=DATA_DIR / "math500_test.json")

band = sorted((r for r in math500 if int(r["level"]) in (4, 5)), key=lambda r: r["unique_id"])
random.Random(SEED).shuffle(band)
items = band[: CFG["n_items"]]

ITEM_SET_HASH = harness.sha256_of(r["unique_id"] for r in items)
print(f"{DATASET}: {len(items)} of {len(band)} available   sha256 {ITEM_SET_HASH[:12]}")

P6_MANIFEST = PAGE_06_DIR / "manifest.json"
SAME_ITEMS_AS_PAGE_6 = None
if P6_MANIFEST.exists():
    p6_hash = json.loads(P6_MANIFEST.read_text()).get("items", {}).get("item_set_sha256")
    SAME_ITEMS_AS_PAGE_6 = p6_hash == ITEM_SET_HASH
    print("page 6's draw:", "the same" if SAME_ITEMS_AS_PAGE_6
          else "DIFFERENT; match n_items, SEED and DATASET")

## 3. When two chains agree

Two answers are the same answer when ch03's `grade_answer` says they are equivalent, so
`\frac{1}{2}` and `0.5` are one vote. Each answer class is labelled with the first form seen.

A chain with no usable answer does not vote. The vote reads each chain with ch03's `number_only`
fallback; grading a single chain uses the default fallback, as page 6 does.

In [ ]:
NO_ANSWER = None
NO_ANSWER_LABEL = "(no usable answer)"


def vote_key(text):
    """What a chain contributes to the tally, or None if it contributed nothing."""
    cand = extract_final_candidate(text, fallback="number_only")
    return cand if cand and cand.strip() else NO_ANSWER


_EQ_CACHE = {}


def equivalent(a, b):
    if a == b:
        return True
    k = (a, b) if a <= b else (b, a)
    if k not in _EQ_CACHE:
        try:
            # grade_answer is not documented as symmetric, so ask it both ways.
            _EQ_CACHE[k] = bool(grade_answer(a, b)) or bool(grade_answer(b, a))
        except Exception:
            _EQ_CACHE[k] = False
    return _EQ_CACHE[k]


def cluster(keys):
    """Group answer strings into equivalence classes, first-seen form as the label.
    Returns (label per input, ordered class labels)."""
    labels, order = [], []
    for k in keys:
        if k is NO_ANSWER:
            labels.append(NO_ANSWER)
            continue
        hit = next((lab for lab in order if equivalent(lab, k)), None)
        if hit is None:
            order.append(k)
            hit = k
        labels.append(hit)
    return labels, order


demo_keys = [r"\frac{1}{2}", "0.5", "1,024", "1024", "3", r"\frac{1}{2}", "0.500"]
lab, order = cluster(demo_keys)
print("clustering, on made-up answers so the rule is visible before it is used:")
for k, l in zip(demo_keys, lab):
    print(f"   {k!r:<16} -> class {l!r}")
print(f"   {len(demo_keys)} answers, {len(set(lab))} distinct by string, {len(order)} classes")

## 4. The tally

One row per problem:

| field | what it is |
|---|---|
| `distinct_answers` | equivalence classes among the chains that answered |
| `answer_entropy` | Shannon entropy over the class frequencies, in bits |
| `plurality_answer` | the class with the most votes, `None` on a tie |
| `plurality_correct` | whether that class is the right answer |
| `any_correct` | whether any chain got it right (coverage) |
| `no_answer_rate` | chains that produced nothing usable |

A tie is not a vote: as in ch04's `self_consistency_vote`, a tie gives no answer and grades wrong.

In [ ]:
def tally(records):
    """One problem, N chains -> the row the figures read."""
    gt = records[0]["ground_truth"]
    labels, _ = cluster([vote_key(r["trace"]) for r in records])

    counts = Counter([l for l in labels if l is not NO_ANSWER])
    n = len(records)
    ranked = counts.most_common()
    top = ranked[0][1] if ranked else 0
    winners = [a for a, c in ranked if c == top]
    tie = len(winners) > 1
    plurality = winners[0] if (winners and not tie) else None
    runner_up = next((c for _, c in ranked[1:]), 0)
    total = sum(counts.values())
    entropy = -sum((c / total) * math.log2(c / total) for c in counts.values()) if total else 0.0

    return dict(
        n_chains=n,
        distinct_answers=len(counts),
        answer_entropy=entropy,
        # 1.0 = every chain said something different, 0.0 = they all agreed
        entropy_normalised=(entropy / math.log2(n)) if n > 1 and total else 0.0,
        plurality_answer=plurality,
        plurality_votes=top,
        plurality_margin=top - runner_up,
        tie=tie,
        plurality_correct=bool(plurality is not None and grade_answer(plurality, gt)),
        any_correct=any(r["correct"] for r in records),
        first_chain_correct=bool(records[0]["correct"]),
        mean_chain_correct=mean([r["correct"] for r in records]),
        no_answer_rate=mean([l is NO_ANSWER for l in labels]),
        truncated_rate=mean([r["truncated"] for r in records]),
        tokens_generated=sum(r["tokens_generated"] for r in records),
        ranked=[(a, c) for a, c in ranked],
    )


def aggregate(rows):
    """Many problems, one configuration -> the point the curves plot."""
    n = len(rows)
    k = sum(r["plurality_correct"] for r in rows)
    lo, hi = harness.wilson(k, n)
    return dict(
        n_items=n,
        plurality_accuracy=k / n, ci_lo=lo, ci_hi=hi,
        coverage=mean([r["any_correct"] for r in rows]),
        single_chain_accuracy=mean([r["mean_chain_correct"] for r in rows]),
        thrown_away=mean([r["any_correct"] and not r["plurality_correct"] for r in rows]),
        mean_distinct=mean([r["distinct_answers"] for r in rows]),
        mean_entropy=mean([r["answer_entropy"] for r in rows]),
        tie_rate=mean([r["tie"] for r in rows]),
        no_answer_rate=mean([r["no_answer_rate"] for r in rows]),
        truncated_rate=mean([r["truncated_rate"] for r in rows]),
        mean_tokens_generated=mean([r["tokens_generated"] for r in rows]),
    )

## Replaying the published run

`REPLAY` reads the published chains from `traces.jsonl` and loads no model. Every cell after the
next one runs the same way in every mode.

In [ ]:
samples, model_meta = {}, {}


def read_traces(path):
    """Every record in the log, with the last write per key winning, so no chain counts twice."""
    latest = {}
    for line in Path(path).open(encoding="utf-8"):
        if not line.strip():
            continue
        r = json.loads(line)
        latest[(r["run_id"], r["config_id"], r["problem_id"], r["sample_index"])] = r
    return list(latest.values())


def config_id(temperature, cap):
    """Names a set of generations. Two askings for the same temperature and the same cap
    share their chains rather than repeating them, which is why a configuration is keyed
    by those two and not by how many chains someone wanted."""
    return f"t{temperature:.2f}-c{cap}"


if RUN_MODE == "REPLAY":
    assert TRACES_PATH.exists() and MANIFEST_PATH.exists(), (
        f"REPLAY reads the published run from {OUT_DIR}/traces.jsonl and manifest.json, "
        f"which come with the notebook download.")
    RUN_MANIFEST = json.loads(MANIFEST_PATH.read_text())
    RUN_ID = RUN_MANIFEST["run_id"]
    assert RUN_MANIFEST["items"]["item_set_sha256"] == ITEM_SET_HASH, \
        "the published run was made on a different set of problems"

    want = {r["unique_id"] for r in items}
    for r in read_traces(TRACES_PATH):
        if r["run_id"] == RUN_ID and r["problem_id"] in want:
            samples.setdefault(r["model_id"], {}).setdefault(r["config_id"], {}) \
                   .setdefault(r["problem_id"], []).append(r)
    for mid in samples:
        for cid in samples[mid]:
            for pid in samples[mid][cid]:
                samples[mid][cid][pid].sort(key=lambda r: r["sample_index"])

    SWEPT = [RUN_MANIFEST["primary_model"]]
    model_meta = {m["id"]: m for m in RUN_MANIFEST["models"]}
    n = sum(len(v) for m in samples.values() for c in m.values() for v in c.values())
    print(f"replaying {RUN_ID}")
    print(f"  {n:,} generations, {len(want)} items, nothing loaded")
    for cid in sorted(samples[SWEPT[0]]):
        k = sorted({len(v) for v in samples[SWEPT[0]][cid].values()})
        print(f"    {cid:<16} {len(samples[SWEPT[0]][cid]):>3} items x {k} chains")
else:
    print(f"{RUN_MODE}: generations are made below")

In [ ]:
def prompt_ids_for(question, tok):
    """The token ids every chain of one problem starts from: ch03's prompt in the chat
    template. The cost probe and the sampler both start here."""
    chat = tok.apply_chat_template([{"role": "user", "content": render_prompt(question)}],
                                   tokenize=False, add_generation_prompt=True)
    return tok(chat, add_special_tokens=False).input_ids


if RUN_MODE != "REPLAY":
    device = get_device()
    DTYPE = harness.pick_dtype(device)
    loaded = harness.load_model(PRIMARY, device=device, dtype=DTYPE, warmup_tokens=16)
    model, tokenizer = loaded["model"], loaded["tokenizer"]
    N_PARAMS, WARMUP_TOK_S = loaded["params"], loaded["tok_per_s"]
    harness.selftest(tokenizer)
    EOS_IDS = harness.end_of_turn_ids(tokenizer)
    SWEPT = [PRIMARY]
    print(f"\nend-of-turn ids: {sorted(EOS_IDS)}")
else:
    device = DTYPE = model = tokenizer = None
    N_PARAMS = WARMUP_TOK_S = EOS_IDS = None
    print("REPLAY: no model loaded, no weights downloaded")

## 5. What this run will cost

Tokens are priced from what page 6's chains actually wrote at each cap. The batched speed is
measured on this machine.

In [ ]:
def page_6_sweep():
    """Page 6's greedy cap sweep on this model and these problems, as {cap: records}.
    Empty when that run is not on disk."""
    path = PAGE_06_DIR / "traces.jsonl"
    by_cap = {}
    if path.exists():
        for line in path.open(encoding="utf-8"):
            r = json.loads(line) if line.strip() else None
            # Page 6's Wait arm carries budget=None and is not part of the sweep.
            if r and r.get("model_id") == PRIMARY and r.get("budget") is not None \
                    and r.get("arm") is None:
                by_cap.setdefault(r["budget"], []).append(r)
    return dict(sorted(by_cap.items()))


P6_SWEEP = page_6_sweep()
P6_SPEND = {b: mean([r["tokens_generated"] for r in rs]) for b, rs in P6_SWEEP.items()}


def spend_at(cap):
    """What one chain at this cap actually writes, from page 6's sweep. Falls back to the
    cap itself, which is the conservative direction for an estimate."""
    if not P6_SPEND:
        return cap
    at_or_below = [b for b in P6_SPEND if b <= cap]
    return P6_SPEND[max(at_or_below)] if at_or_below else min(cap, min(P6_SPEND.values()))


def plan_cost():
    """(tokens, breakdown) for what the cells below will generate, ignoring anything
    already on disk."""
    n, rows = len(items), []
    rows.append(("the pile, %d chains at cap %d" % (POOL_N, CHAIN_CAP),
                 POOL_N * spend_at(CHAIN_CAP) * n))
    hot = [t for t in TEMPERATURES if t > 0]
    rows.append(("diversity, %d extra temperatures x %d chains x %d items"
                 % (max(len(hot) - 1, 0), DIVERSITY_N, DIVERSITY_ITEMS),
                 max(len(hot) - 1, 0) * DIVERSITY_N * spend_at(CHAIN_CAP) * DIVERSITY_ITEMS))
    rows.append(("the temperature-0 line, one chain per item", spend_at(CHAIN_CAP) * n))
    rows.append(("the split axis, N in %s" % SPLIT_N,
                 sum(k * spend_at(B_TOTAL // k) for k in SPLIT_N if k != 1) * n))
    return sum(v for _, v in rows), rows


TOTAL_TOKENS, COST_ROWS = plan_cost()
for what, v in COST_ROWS:
    print(f"  {what:<52}{v / 1e6:>7.2f} M")
print(f"  {'total':<52}{TOTAL_TOKENS / 1e6:>7.2f} M tokens")

BATCH_TOK_S = None
if RUN_MODE != "REPLAY":
    import time as _time
    _inp = torch.tensor([prompt_ids_for(items[0]["problem"], tokenizer)], device=model.device)
    _t0 = _time.time()
    with torch.inference_mode():
        _out = model.generate(input_ids=_inp, attention_mask=torch.ones_like(_inp),
                              max_new_tokens=64, do_sample=True, temperature=0.9,
                              top_p=TOP_P, top_k=None, num_return_sequences=GROUP,
                              pad_token_id=tokenizer.pad_token_id
                              if tokenizer.pad_token_id is not None else tokenizer.eos_token_id)
    BATCH_TOK_S = (GROUP * 64) / max(_time.time() - _t0, 1e-6)
    print(f"\nmeasured on this machine: {BATCH_TOK_S:,.0f} tok/s at a group of {GROUP}, "
          f"against {WARMUP_TOK_S:,.0f} tok/s one at a time")
    print(f"-> about {TOTAL_TOKENS / BATCH_TOK_S / 3600:.1f} h batched, "
          f"{TOTAL_TOKENS / max(WARMUP_TOK_S, 1e-6) / 3600:.1f} h unbatched")
    print("   Whatever is already in traces.jsonl comes off both figures.")
    del _out

## 6. One group of chains

1. **Sample** up to `GROUP` chains from one prompt in one `generate` call, at `temperature` with
   `top_p` = 0.95.
2. **Force** any chain that hit its cap: append `Final answer: \boxed{` and generate greedily
   until the brace closes. At the pile's cap most chains never need this.
3. **Grade** with ch03's grader.

A batch pads every row to the longest, so a chain counts as finished on its own only if it wrote
an end-of-turn token, and its trace ends there.

Each group's seed is recorded on its chains (`batch_seed`, `group_index`), so a group regenerates
exactly on the same hardware and software.

In [ ]:
FORCE_STEM = "\n\nFinal answer: \\boxed{"      # the brace we open and it must close


def group_seed(problem_id, cid, group_index):
    """A stable 63-bit seed per (problem, configuration, group). Recorded on every record
    in the group, so the group regenerates exactly."""
    return int(harness.sha256_of([str(SEED), problem_id, cid, str(group_index)])[:15], 16)


@torch.inference_mode()
def sample_group(question, cap, temperature, k, seed, mdl, tok, allowance=ANSWER_ALLOWANCE):
    """k chains of one problem, in one generate call. Returns k dicts shaped like page 6's."""
    assert k >= 1, f"asked for {k} chains"
    # `num_return_sequences` needs sampling, so several greedy chains (section 9's check
    # that greedy chains are identical) are made one at a time, independently.
    if temperature == 0.0 and k > 1:
        out = []
        for _ in range(k):
            out.extend(sample_group(question, cap, temperature, 1, seed, mdl, tok, allowance))
        return out

    prompt_ids = prompt_ids_for(question, tok)
    inp = torch.tensor([prompt_ids], device=mdl.device)
    sampling = temperature > 0
    if sampling:
        torch.manual_seed(seed)
    out = mdl.generate(
        input_ids=inp, attention_mask=torch.ones_like(inp), max_new_tokens=cap,
        # Every decoding parameter is set, overriding the checkpoint's own defaults.
        do_sample=sampling,
        temperature=temperature if sampling else None,
        top_p=TOP_P if sampling else None, top_k=None,
        num_return_sequences=k,
        pad_token_id=tok.pad_token_id if tok.pad_token_id is not None else tok.eos_token_id)

    chains = []
    for row in out:
        gen = row[inp.shape[1]:].tolist()
        # The chain ends at its first end-of-turn token; the rest is batch padding.
        stop = next((i for i, t in enumerate(gen) if t in EOS_IDS), None)
        hit_cap = stop is None
        gen = gen[:stop] if stop is not None else gen[:cap]

        own_text = tok.decode(gen, skip_special_tokens=True)
        own_pieces = harness.incremental_token_strings(gen, tok)
        assert "".join(own_pieces) == own_text

        if not hit_cap:
            chains.append(dict(trace=own_text, tokens=own_pieces, own_tokens=len(gen),
                               answer_tokens=0, tokens_generated=len(gen),
                               truncated=False, forced=False, forced_suffix="",
                               answer_closed=None, cut_index=None, answer_start=None))
            continue

        suffix_ids = tok(FORCE_STEM, add_special_tokens=False).input_ids
        suffix_pieces = harness.incremental_token_strings(suffix_ids, tok)
        assert "".join(suffix_pieces) == FORCE_STEM, "the forced suffix does not round-trip"
        # The forced answer is greedy at every temperature.
        ans_ids = harness.continue_ids(prompt_ids + gen + suffix_ids, mdl, tok, allowance)
        while ans_ids and ans_ids[-1] in EOS_IDS:
            ans_ids.pop()
        kept, closed = harness.cut_at_boxed_close(harness.incremental_token_strings(ans_ids, tok))

        tokens = own_pieces + suffix_pieces + kept
        chains.append(dict(trace="".join(tokens), tokens=tokens, own_tokens=len(gen),
                           answer_tokens=len(kept),
                           tokens_generated=len(gen) + len(kept),   # our suffix is not its cost
                           truncated=True, forced=True, forced_suffix=FORCE_STEM,
                           answer_closed=closed, cut_index=len(own_pieces),
                           answer_start=len(own_pieces) + len(suffix_pieces)))
    return chains

In [ ]:
def render_chain(r, width=88, head=3, tail=2):
    """One chain the way the page draws it."""
    own = "".join(r["tokens"][: r["cut_index"]]) if r["forced"] else r["trace"]
    lines = textwrap.wrap(own.strip().replace("\n", " "), width) or ["(nothing generated)"]
    if len(lines) > head + tail:
        lines = lines[:head] + [f"… [{len(lines) - head - tail} lines elided] …"] + lines[-tail:]
    out = [f"    {ln}" for ln in lines]
    if r["forced"]:
        out[-1] += " ▌"
        out.append(f"    >> Final answer: \\boxed{{{''.join(r['tokens'][r['answer_start']:])}")
    return "\n".join(out)


if RUN_MODE != "REPLAY":
    demo = items[0]
    print(textwrap.fill("Q: " + demo["problem"], 92))
    print("ground truth:", demo["answer"])
    for t in (0.0, TEMPERATURES[-1]):
        cid = config_id(t, CHAIN_CAP)
        got = sample_group(demo["problem"], CHAIN_CAP, t, 3,
                           group_seed(demo["unique_id"], cid, 0), model, tokenizer)
        print(f"\n{'=' * 96}\ntemperature {t}, three chains:")
        for k, g in enumerate(got):
            a = extract_final_candidate(g["trace"])
            print(f"\n  chain {k}   {g['own_tokens']} tok   answer={a!r}   "
                  f"{'CORRECT' if grade_answer(a, demo['answer']) else 'wrong'}")
            print(render_chain(g))
    print("\nAt temperature 0 the three chains are one chain. Above it they are three.")

## 7. The sweep machinery

`ensure_samples` generates *k* chains at one temperature and cap, skipping any already in
`traces.jsonl`. Asking twice for the same temperature and cap reuses the same chains.

At temperature 0 one chain is generated and copied (`replicated_from`), since greedy decoding
repeats itself; section 9 checks this on the first few problems. Changing the model, problems,
cap, pile size or top-p changes `RUN_ID`, and traces from other runs are ignored.

In [ ]:
if RUN_MODE != "REPLAY":
    RUN_ID = (f"{RUN_MODE.lower()}-{DATASET}-n{len(items)}-{ITEM_SET_HASH[:8]}"
              f"-cap{CHAIN_CAP}-pool{POOL_N}-p{TOP_P}-{str(DTYPE).rsplit('.', 1)[-1]}"
              f"-{device.type}")
    STORE = harness.TraceStore(TRACES_PATH,
                               RUN_ID, ("model_id", "config_id", "problem_id", "sample_index"))
    STORE.report()

    # Verified on the first few items, where the whole group is generated rather than copied.
    T0_VERIFY_ITEMS = min(5, len(items))


    def ensure_samples(temperature, cap, k, rows, label=""):
        cid = config_id(temperature, cap)
        out, n_gen, n_rep = {}, 0, 0
        replicate = temperature == 0.0

        bar = harness.progress(rows, desc=f"{label or cid:<28}", unit="item")
        for i, row in enumerate(bar):
            pid = row["unique_id"]
            got = [STORE.get(PRIMARY, cid, pid, s) for s in range(k)]
            if all(h is not None for h in got):
                out[pid] = got
                continue

            # Only the groups that are actually missing.
            step = 1 if replicate else GROUP
            # Greedy: one real chain, copied. Verified in full on the first few items.
            n_real = k if (replicate and i < T0_VERIFY_ITEMS) else (1 if replicate else k)
            for start in range(0, n_real, step):
                size = min(step, n_real - start)
                if all(got[s] is not None for s in range(start, start + size)):
                    continue                          # this group is already on disk
                g_index = start // step
                seed = group_seed(pid, cid, g_index)
                made = sample_group(row["problem"], cap, temperature, size, seed,
                                    model, tokenizer)
                n_gen += size
                for j, g in enumerate(made):
                    s = start + j
                    extracted = extract_final_candidate(g["trace"])
                    got[s] = STORE.append({
                        "model_id": PRIMARY, "config_id": cid,
                        "problem_id": pid, "sample_index": s,
                        "group_index": g_index, "batch_seed": seed, "batch_size": size,
                        "level": int(row["level"]), "subject": row["subject"],
                        "question": row["problem"], "ground_truth": row["answer"],
                        "temperature": temperature,
                        "top_p": TOP_P if temperature > 0 else None,
                        "chain_budget": cap, "budget_total": B_TOTAL,
                        "trace": g["trace"], "tokens": g["tokens"],
                        "cut_index": g["cut_index"], "answer_start": g["answer_start"],
                        "truncated": g["truncated"], "forced": g["forced"],
                        "forced_suffix": g["forced_suffix"],
                        "answer_closed": g["answer_closed"],
                        "extracted_answer": extracted,
                        "vote_key": vote_key(g["trace"]),
                        "correct": bool(grade_answer(extracted, row["answer"])),
                        "own_tokens": g["own_tokens"], "answer_tokens": g["answer_tokens"],
                        "tokens_generated": g["tokens_generated"],
                        "replicated_from": None,
                    })
            # Greedy only: the slots past the one real chain are copies of it.
            for s in range(n_real, k):
                if got[s] is None:
                    got[s] = STORE.append(dict(got[0], sample_index=s, batch_seed=None,
                                               replicated_from=0))
                    n_rep += 1
            assert all(g is not None for g in got), \
                f"{cid} {pid}: {sum(g is None for g in got)} of {k} chains never filled"
            out[pid] = got
            bar.set_postfix(gen=n_gen, repl=n_rep)

        samples.setdefault(PRIMARY, {}).setdefault(cid, {}).update(out)
        return out
else:
    def ensure_samples(*a, **k):
        raise RuntimeError("REPLAY does not generate; the published traces are the run")

## 8. The pile, the diversity family, and the split

The diversity family runs first, on the first `DIVERSITY_ITEMS` problems, and picks the
temperature the pile is sampled at. Then the pile on every problem, then the split axis.

In [ ]:
if RUN_MODE != "REPLAY":
    for t in [x for x in TEMPERATURES if x > 0]:
        ensure_samples(t, CHAIN_CAP, DIVERSITY_N, items[:DIVERSITY_ITEMS],
                       label=f"diversity T={t}")
    ensure_samples(0.0, CHAIN_CAP, DIVERSITY_N, items[:DIVERSITY_ITEMS],
                   label="diversity T=0.0")
    print("\ndiversity family complete")

PRIMARY_SAMPLES = samples[SWEPT[0]]


def rows_at(cid, k=None, only=None):
    """The per-problem tallies at one configuration. `k` trims each vote to its first k
    chains; `only` restricts to a set of problem ids."""
    out = []
    for p in items:
        if only is not None and p["unique_id"] not in only:
            continue
        rs = PRIMARY_SAMPLES.get(cid, {}).get(p["unique_id"])
        if rs and (k is None or len(rs) >= k):
            out.append(tally(rs[:k] if k else rs))
    return out


# The pile's temperature: the one whose vote solves the most problems, ties to the lower.
DIV_IDS = {p["unique_id"] for p in items[:DIVERSITY_ITEMS]}
D_AGG = {}
for t in TEMPERATURES:
    rs = rows_at(config_id(t, CHAIN_CAP), DIVERSITY_N, only=DIV_IDS)
    if rs:
        D_AGG[t] = aggregate(rs)
T_BEST = max(D_AGG, key=lambda t: (D_AGG[t]["plurality_accuracy"], -t))

print(f"diversity family: N={DIVERSITY_N} chains on {len(DIV_IDS)} items")
print(f"  {'T':>5}{'distinct':>10}{'entropy':>9}{'one chain':>11}{'vote':>7}{'coverage':>10}")
for t, a in sorted(D_AGG.items()):
    print(f"  {t:>5}{a['mean_distinct']:>10.1f}{a['mean_entropy']:>9.2f}"
          f"{a['single_chain_accuracy']:>11.0%}{a['plurality_accuracy']:>7.0%}"
          f"{a['coverage']:>10.0%}")
print(f"\n-> the pile is generated at T={T_BEST}")

In [ ]:
if RUN_MODE != "REPLAY":
    ensure_samples(T_BEST, CHAIN_CAP, POOL_N, items, label=f"the pile T={T_BEST}")
    ensure_samples(0.0, CHAIN_CAP, POOL_N, items, label="the pile T=0")
    print("\npile complete")

POOL_CONFIG = config_id(T_BEST, CHAIN_CAP)
pile = PRIMARY_SAMPLES.get(POOL_CONFIG, {})

# One tally per problem over the whole pile.
POOL_ROWS = {p["unique_id"]: tally(pile[p["unique_id"]]) for p in items}
print(f"pile {POOL_CONFIG}: {POOL_N} chains x {len(POOL_ROWS)} problems, N axis {N_AXIS}")

In [ ]:
if RUN_MODE != "REPLAY":
    for n in SPLIT_N:
        if n == 1:
            continue                      # N=1 of the split axis IS the pile's first chain
        ensure_samples(T_BEST, B_TOTAL // n, n, items, label=f"split N={n}")
    print("\nsplit axis complete")

## 9. Checks on the run

1. Every chain rebuilds exactly from its stored tokens.
2. The split's chains stay within the shared budget `B`; the forced answers are reported on top.
3. Sampled chains differ, and greedy chains are identical.

In [ ]:
all_records = [r for cid in PRIMARY_SAMPLES for rs in PRIMARY_SAMPLES[cid].values() for r in rs]
generated = [r for r in all_records if r.get("replicated_from") is None]
print(f"{len(all_records):,} records  ({len(generated):,} generations, "
      f"{len(all_records) - len(generated):,} replicated at temperature 0)\n")

bad = [r for r in all_records if "".join(r["tokens"]) != r["trace"]]
print(f"1. traces reconstruct from tokens .......... {len(bad)} violations")
assert not bad, "a trace's token strings do not join back to its text"

print("\n2. what each configuration spent per problem:")
print(f"   {'config':<16}{'chains':>7}{'thinking':>10}{'forced':>8}{'total':>9}{'of B':>7}")
BUDGET_AUDIT = []
for cid in sorted(PRIMARY_SAMPLES):
    by_item = PRIMARY_SAMPLES[cid]
    cap = next(iter(by_item.values()))[0]["chain_budget"]
    is_split = cap < CHAIN_CAP
    k = max(1, B_TOTAL // cap) if is_split else median([len(v) for v in by_item.values()])
    think = mean([sum(r["own_tokens"] for r in rs[:k]) for rs in by_item.values()])
    ans = mean([sum(r["answer_tokens"] for r in rs[:k]) for rs in by_item.values()])
    BUDGET_AUDIT.append(dict(config_id=cid, axis="split" if is_split else "pile",
                             chains=k, mean_thinking=think, mean_forced=ans,
                             mean_total=think + ans, of_budget=(think + ans) / B_TOTAL))
    print(f"   {cid:<16}{k:>7}{think:>10,.0f}{ans:>8,.0f}{think + ans:>9,.0f}"
          f"{(think + ans) / B_TOTAL:>7.2f}x   {'split' if is_split else 'pile'}")
over = [a for a in BUDGET_AUDIT if a["axis"] == "split" and a["mean_thinking"] > B_TOTAL + 1]
assert not over, f"{[a['config_id'] for a in over]} wrote more than B tokens"

In [ ]:
print("3. what sampling did, over generated chains only:")
DIVERSITY_CHECK, T0_IDENTICAL = [], None
for cid in sorted(PRIMARY_SAMPLES):
    real = {pid: [r for r in rs if r.get("replicated_from") is None]
            for pid, rs in PRIMARY_SAMPLES[cid].items()}
    real = {pid: rs for pid, rs in real.items() if len(rs) > 1}
    if not real:
        print(f"   {cid:<16} one generated chain per item; nothing to compare")
        continue
    t = next(iter(real.values()))[0]["temperature"]
    ident = mean([len({r["trace"] for r in rs}) == 1 for rs in real.values()])
    DIVERSITY_CHECK.append(dict(config_id=cid, temperature=t, n_items_compared=len(real),
                                all_identical_rate=ident))
    print(f"   {cid:<16} T={t:<5} {len(real):>3} items compared, all chains identical on "
          f"{ident:>4.0%}")
    if t > 0:
        assert ident < 1.0, f"{cid} sampled at T={t} but every chain is identical"
    else:
        T0_IDENTICAL = ident if T0_IDENTICAL is None else min(T0_IDENTICAL, ident)
        assert ident == 1.0, "greedy decoding produced different chains from one prompt"

n_replicated = sum(1 for r in all_records if r.get("replicated_from") is not None)
if T0_IDENTICAL is None and n_replicated:
    print(f"\n   !! {n_replicated:,} temperature-0 copies, none checked against a real chain")

INTEGRITY = dict(n_records=len(all_records), n_generated=len(generated),
                 budget_audit=BUDGET_AUDIT, diversity_check=DIVERSITY_CHECK,
                 t0_identical_rate=T0_IDENTICAL)

# Part 2: the results

Every figure and number the page shows, computed from part 1 and saved to
`out/page_07/results.json`.

## 10. Figure 1: one problem's tally

One problem's `POOL_N` chains, grouped by answer and ranked by votes. The problem is picked from
the pile: the vote is right and at least three distinct answers appear. Answers given once stay
as their own rows.

In [ ]:
candidates = []
for pid, row in POOL_ROWS.items():
    if not row["plurality_correct"] or row["distinct_answers"] < 3:
        continue                     # a right vote, and at least three answers
    singletons = sum(1 for _, c in row["ranked"] if c == 1)
    candidates.append((-singletons, -row["plurality_margin"], pid))
candidates.sort()

SPECIMEN_IS_FALLBACK = False
if candidates:
    SPECIMEN_ID = candidates[0][2]
else:
    SPECIMEN_IS_FALLBACK = True
    SPECIMEN_ID = max(POOL_ROWS, key=lambda p: POOL_ROWS[p]["distinct_answers"])
    print("WARNING: no problem has a right vote and three distinct answers; figure 1 is flagged\n")

spec = POOL_ROWS[SPECIMEN_ID]
spec_chains = PRIMARY_SAMPLES[POOL_CONFIG][SPECIMEN_ID]
print(textwrap.fill("Q: " + spec_chains[0]["question"], 92))
print("ground truth:", spec_chains[0]["ground_truth"])
print(f"\n{spec['n_chains']} chains at T={T_BEST}, {spec['distinct_answers']} distinct answers\n")
print(f"  {'answer':<26}{'votes':>6}")
for ans, c in spec["ranked"]:
    mark = "  <- correct" if grade_answer(ans, spec_chains[0]["ground_truth"]) else ""
    print(f"  {str(ans)[:25]:<26}{c:>6}{mark}")
if spec["no_answer_rate"]:
    print(f"  {NO_ANSWER_LABEL:<26}{round(spec['no_answer_rate'] * spec['n_chains']):>6}")
print(f"\nvote -> {spec['plurality_answer']!r}   "
      f"{'CORRECT' if spec['plurality_correct'] else 'wrong'}   margin {spec['plurality_margin']}")

# The same tally across every problem.
SHAPE = dict(
    mean_distinct=mean([r["distinct_answers"] for r in POOL_ROWS.values()]),
    singleton_share=mean([sum(1 for _, c in r["ranked"] if c == 1) / max(len(r["ranked"]), 1)
                          for r in POOL_ROWS.values()]),
    mean_top_votes=mean([r["plurality_votes"] for r in POOL_ROWS.values()]),
    n_chains=POOL_N)
print(f"\nacross all {len(POOL_ROWS)} problems: {SHAPE['mean_distinct']:.1f} distinct answers "
      f"per {POOL_N} chains,\n{SHAPE['singleton_share']:.0%} of answer classes appear exactly "
      f"once, and the top class holds {SHAPE['mean_top_votes']:.1f} votes.")

In [ ]:
# The chains behind the top three answers.
labels, _ = cluster([vote_key(r["trace"]) for r in spec_chains])
groups = {}
for lab, r in zip(labels, spec_chains):
    groups.setdefault(NO_ANSWER_LABEL if lab is NO_ANSWER else lab, []).append(r)

for ans, rs in sorted(groups.items(), key=lambda kv: -len(kv[1]))[:3]:
    ok = ans != NO_ANSWER_LABEL and bool(grade_answer(ans, spec_chains[0]["ground_truth"]))
    print(f"\n{'=' * 96}\nanswer {ans!r}  x{len(rs)}  {'CORRECT' if ok else 'wrong'}")
    for r in rs[:2]:
        print(f"\n  chain {r['sample_index']}   {r['own_tokens']} tok   "
              f"truncated={r['truncated']}")
        print(render_chain(r))

In [ ]:
import matplotlib.pyplot as plt

ranked = spec["ranked"]
counts = [c for _, c in ranked]
is_right = [bool(grade_answer(a, spec_chains[0]["ground_truth"])) for a, _ in ranked]

fig, ax = plt.subplots(figsize=(7.4, max(2.6, 0.32 * len(ranked) + 1.4)))
ax.barh(range(len(ranked)), counts,
        color=["#3b4a6b" if r else "#c4c0b8" for r in is_right], height=0.66)
ax.set_yticks(range(len(ranked)))
ax.set_yticklabels([str(a)[:18] for a, _ in ranked], fontsize=8)
ax.invert_yaxis()
ax.set_xlabel("chains landing on this answer")
ax.set_xticks(range(0, max(counts) + 1))
ax.spines[["top", "right"]].set_visible(False)
ax.set_title(f"{spec['n_chains']} chains, T={T_BEST}   ·   {spec['distinct_answers']} distinct "
             f"answers, {sum(1 for c in counts if c == 1)} of them once",
             fontsize=9, loc="left", color="#555")
fig.tight_layout()
fig.savefig(OUT_DIR / "figure_1_tally.png", dpi=160)
plt.show()

## 11. Figure 2: accuracy against N

A point at N is the average over `N_RESAMPLES` random N-chain subsets of the pile. Chain length
is fixed, so only the number of votes changes. At temperature 0 the line is flat: N copies of one
greedy chain vote as one.

The vote is compared with one sampled chain and with one greedy chain (bare chain of thought),
each with an exact McNemar test.

In [ ]:
def subsets_for(pid, n, repeats, size=None):
    """The draws a point averages over. Exhaustive when the configuration holds exactly
    n chains, because then there is only one subset to draw."""
    size = POOL_N if size is None else size
    if n >= size:
        return [list(range(size))]
    rng = random.Random(harness.sha256_of([str(SEED), pid, str(n), str(size)])[:15])
    return [rng.sample(range(size), n) for _ in range(repeats)]


def curve_point(cid, n):
    """One point on either axis, computed the same way on both.

    The mean over random n-subsets of whatever chains the configuration holds. Where a
    configuration holds exactly n, `subsets_for` returns the single full subset and this
    degenerates to one tally, so the two axes cannot disagree at a shared configuration.
    """
    per_item = []
    for p in items:
        rs = PRIMARY_SAMPLES.get(cid, {}).get(p["unique_id"])
        if not rs or len(rs) < n:
            continue
        ts = [tally([rs[i] for i in idx])
              for idx in subsets_for(p["unique_id"], n, N_RESAMPLES, len(rs))]
        per_item.append({k: mean([t[k] for t in ts]) for k in
                         ("plurality_correct", "any_correct", "tie", "mean_chain_correct",
                          "distinct_answers", "tokens_generated", "truncated_rate")})
    if not per_item:
        return None
    m = lambda k: mean([x[k] for x in per_item])
    return dict(n=n, n_items=len(per_item),
                accuracy=m("plurality_correct"), coverage=m("any_correct"),
                tie_rate=m("tie"), single_chain_accuracy=m("mean_chain_correct"),
                mean_distinct=m("distinct_answers"),
                # tokens actually written, not n times the cap
                mean_tokens=m("tokens_generated"),
                truncated_rate=m("truncated_rate"),
                n_draws=len(per_item) * N_RESAMPLES)


def pile_curve(cid):
    return [pt for n in N_AXIS if (pt := curve_point(cid, n))]


PILE = {}
for t in sorted({T_BEST, 0.0}):
    cid = config_id(t, CHAIN_CAP)
    if cid in PRIMARY_SAMPLES:
        PILE[t] = pile_curve(cid)

for t, pts in PILE.items():
    print(f"\nT={t}, {POOL_N} chains at a {CHAIN_CAP}-token cap, n={len(items)} items")
    print(f"  {'N':>4}{'vote':>8}{'coverage':>11}{'ties':>7}{'draws':>9}")
    for p in pts:
        print(f"  {p['n']:>4}{p['accuracy']:>8.1%}{p['coverage']:>11.1%}"
              f"{p['tie_rate']:>7.0%}{p['n_draws']:>9,}")

# The vote against one sampled chain, paired.
hot = PILE[T_BEST]
one = {pid: row["first_chain_correct"] for pid, row in POOL_ROWS.items()}
full = {pid: row["plurality_correct"] for pid, row in POOL_ROWS.items()}
fixed, broken, p_value = harness.mcnemar(full, one)
GAIN_PTS = (mean(list(full.values())) - mean(list(one.values()))) * 100
VOTE_BEATS_ONE_CHAIN = GAIN_PTS > 0 and p_value < 0.05

print(f"\none sampled chain      {mean(list(one.values())):.0%}")
print(f"the vote over {POOL_N:<2}     {mean(list(full.values())):.0%}")
print(f"the gain               {GAIN_PTS:+.0f} points   ({fixed} fixed, {broken} broken, "
      f"exact McNemar p={p_value:.4f})")

# The vote against one greedy chain (bare chain of thought), paired. This is the page's contrast.
GREEDY_CONFIG = config_id(0.0, CHAIN_CAP)
BARE, BARE_GAIN_PTS, VOTE_BEATS_BARE_COT = None, None, False
if GREEDY_CONFIG in PRIMARY_SAMPLES:
    bare = {pid: PRIMARY_SAMPLES[GREEDY_CONFIG][pid][0]["correct"]
            for pid in full if pid in PRIMARY_SAMPLES[GREEDY_CONFIG]}
    paired = {pid: full[pid] for pid in bare}
    b_fixed, b_broken, b_p = harness.mcnemar(paired, bare)
    BARE_GAIN_PTS = (mean(list(paired.values())) - mean(list(bare.values()))) * 100
    VOTE_BEATS_BARE_COT = BARE_GAIN_PTS > 0 and b_p < 0.05
    BARE = dict(bare_cot=mean(list(bare.values())), vote=mean(list(paired.values())),
                gain_pts=BARE_GAIN_PTS, p=b_p, fixed=b_fixed, broken=b_broken,
                n_items=len(bare), config_id=GREEDY_CONFIG)
    print(f"\nbare chain of thought  {BARE['bare_cot']:.0%}   (one greedy chain, the same "
          f"{CHAIN_CAP}-token cap)")
    print(f"the vote over {POOL_N:<2}     {BARE['vote']:.0%}")
    print(f"the gain               {BARE_GAIN_PTS:+.0f} points   ({b_fixed} fixed, "
          f"{b_broken} broken, exact McNemar p={b_p:.4f})")
else:
    print(f"\nno greedy pile at {GREEDY_CONFIG}: no contrast against bare chain of thought")

for label, ok in (("one sampled chain", VOTE_BEATS_ONE_CHAIN),
                  ("bare chain of thought", VOTE_BEATS_BARE_COT)):
    if not ok:
        print(f"\n!! the vote does not beat {label} at p<0.05")

## 12. Figure 2, right: a fixed total budget

N chains sharing `B` tokens, `B/N` each.

In [ ]:
SPLIT = []
for n in SPLIT_N:
    cid = POOL_CONFIG if n == 1 else config_id(T_BEST, B_TOTAL // n)
    pt = curve_point(cid, n)          # the same estimator the pile uses
    if pt is None:
        continue
    SPLIT.append(dict(chain_budget=B_TOTAL // n, temperature=T_BEST,
                      # page 6's greedy accuracy at this length, for the site's tests
                      page_06_greedy=(mean([r["correct"] for r in P6_SWEEP[B_TOTAL // n]])
                                      if B_TOTAL // n in P6_SWEEP else None),
                      plurality_accuracy=pt["accuracy"], **pt))

print(f"one budget of {B_TOTAL} tokens, split N ways, T={T_BEST}, n={len(items)}")
print(f"  {'N':>4}{'cap each':>10}{'spent':>9}{'the vote':>10}{'coverage':>10}"
      f"{'cut off':>9}")
for s in SPLIT:
    print(f"  {s['n']:>4}{s['chain_budget']:>10}{s['mean_tokens']:>9,.0f}"
          f"{s['plurality_accuracy']:>10.0%}{s['coverage']:>10.0%}"
          f"{s['truncated_rate']:>9.0%}")

peak = max(SPLIT, key=lambda s: s["plurality_accuracy"])
SPLIT_PAYS = peak["n"] > 1
print(f"\nthe best split is N={peak['n']} ({peak['plurality_accuracy']:.0%})")

# The same N with and without the shared budget, to separate the cap from the split.
print("\nthe same N, capped against uncapped:")
print(f"  {'N':>4}{'capped':>9}{'spent':>9}{'cut off':>9}   |{'uncapped':>10}{'spent':>9}")
for sp in SPLIT:
    up = next((p for p in (PILE.get(T_BEST) or []) if p["n"] == sp["n"]), None)
    if up is None:
        continue
    print(f"  {sp['n']:>4}{sp['plurality_accuracy']:>9.0%}{sp['mean_tokens']:>9,.0f}"
          f"{sp['truncated_rate']:>9.0%}   |{up['accuracy']:>10.0%}{up['mean_tokens']:>9,.0f}")


In [ ]:
fig, (ax, axs) = plt.subplots(1, 2, figsize=(11.2, 4.3),
                              gridspec_kw=dict(width_ratios=[1.35, 1], wspace=0.28))
INK = {0.0: "#9aa0ab"}
for t in sorted(PILE):
    pts = PILE[t]
    ax.plot([p["n"] for p in pts], [p["accuracy"] * 100 for p in pts], marker="o", ms=4.5, lw=2,
            color=INK.get(t, "#3b4a6b"),
            label=f"the vote, T={t}" + ("  (one voter, N times)" if t == 0 else ""))
ax.plot([p["n"] for p in hot], [p["coverage"] * 100 for p in hot], ls="--", lw=1.2,
        color="#a2704a", label="the right answer is somewhere in the pile")
ax.set_xscale("log", base=2)
ax.set_xticks(N_AXIS)
ax.set_xticklabels([str(n) for n in N_AXIS])
ax.set_xlabel(f"chains, each free to run to its own end (cap {CHAIN_CAP})")
ax.set_ylabel("share of problems solved (%)")
ax.set_ylim(0, min(100, max(p["coverage"] for p in hot) * 100 + 12))
ax.grid(axis="y", alpha=0.25)
ax.spines[["top", "right"]].set_visible(False)
ax.legend(frameon=False, fontsize=8, loc="upper left")
ax.set_title("what the vote buys", fontsize=9, loc="left", color="#555")

axs.plot([s["n"] for s in SPLIT], [s["plurality_accuracy"] * 100 for s in SPLIT],
         marker="o", ms=4.5, lw=2, color="#b23", label=f"the vote, T={T_BEST}")
axs.set_xscale("log", base=2)
axs.set_xticks(SPLIT_N)
axs.set_xticklabels([f"{n}\n{B_TOTAL // n}" for n in SPLIT_N], fontsize=8)
axs.set_xlabel(f"chains, and tokens each (total held at {B_TOTAL})")
axs.set_ylim(0, min(100, max(p["coverage"] for p in hot) * 100 + 12))
axs.grid(axis="y", alpha=0.25)
axs.spines[["top", "right"]].set_visible(False)
axs.legend(frameon=False, fontsize=8, loc="upper right")
axs.set_title("what it costs at a fixed budget", fontsize=9, loc="left", color="#555")

fig.suptitle(f"{SWEPT[0].split('/')[-1]}, {DATASET}, n={len(items)}", fontsize=10, x=0.02,
             ha="left", color="#333")
fig.tight_layout()
fig.savefig(OUT_DIR / "figure_2_vote.png", dpi=160)
plt.show()

## 13. The number page 8 starts from

The share of problems where some chain was right but the vote was not: coverage minus the vote's
accuracy, on the pile.

In [ ]:
A_POOL = aggregate(list(POOL_ROWS.values()))

HANDOFF = dict(config_id=POOL_CONFIG, n_chains=POOL_N, temperature=T_BEST,
               chain_cap=CHAIN_CAP, n_items=A_POOL["n_items"],
               single_chain_accuracy=A_POOL["single_chain_accuracy"],
               plurality_accuracy=A_POOL["plurality_accuracy"],
               coverage=A_POOL["coverage"], thrown_away=A_POOL["thrown_away"],
               tie_rate=A_POOL["tie_rate"], gain_pts=GAIN_PTS, gain_p=p_value,
               gain_fixed=fixed, gain_broken=broken)

print(f"{POOL_N} chains at T={T_BEST}, {A_POOL['n_items']} problems: the right answer is in the pile "
      f"on {A_POOL['coverage']:.0%}, the vote picks it on {A_POOL['plurality_accuracy']:.0%}")
print(f"\n-> on {A_POOL['thrown_away']:.0%} of problems the model produced the right answer and "
      f"the vote threw it away")
print(f"   ({A_POOL['tie_rate']:.0%} of votes were ties, which count as thrown away)")
if A_POOL["thrown_away"] < 0.05:
    print("\n!! under five points between coverage and the vote")

## 14. Save the results

`results.json` holds every number above and the `claims` the page checks before it renders;
`manifest.json` records what was run.

In [ ]:
CLAIMS = {
    "vote_beats_bare_cot": VOTE_BEATS_BARE_COT,
    "vote_beats_one_chain": VOTE_BEATS_ONE_CHAIN,
    "coverage_exceeds_vote": A_POOL["thrown_away"] > 0.0,
    "t0_vote_is_a_noop": T0_IDENTICAL in (None, 1.0),
    "diversity_is_measured": len(D_AGG) >= 2,
    "specimen_usable": not SPECIMEN_IS_FALLBACK,
    "errors_are_diffuse": SHAPE["singleton_share"] > 0.5,
    "splitting_a_fixed_budget_pays": SPLIT_PAYS,
    "pool_supports_page_8": A_POOL["thrown_away"] >= 0.05,
}

harness.write_manifest(
    MANIFEST_PATH, page=7, notebook="page_07_frequency_voting.ipynb", run_id=RUN_ID, run_mode=RUN_MODE,
    primary_model=SWEPT[0],
    models=[model_meta.get(m, {"id": m}) for m in SWEPT],
    decoding={"strategy": "sampled above T=0, greedy at T=0", "top_p": TOP_P,
              "temperatures": TEMPERATURES, "pile_temperature": T_BEST,
              "seed_rule": "sha256(SEED, problem_id, config_id, group_index), one per group",
              "group_size": GROUP,
              "batching": "the chains of one problem are one generate call with "
                          "num_return_sequences; temperature 0 is generated unbatched",
              "forced_answer_decoding": "greedy at every temperature"},
    pile={"chain_cap": CHAIN_CAP, "n_chains": POOL_N, "config_id": POOL_CONFIG,
          "n_axis": N_AXIS, "n_resamples": N_RESAMPLES,
          "axis_rule": "a point at N is the mean over random N-subsets of the same pile"},
    split={"budget_total": B_TOTAL, "n_grid": SPLIT_N,
           "chain_budgets": {str(n): B_TOTAL // n for n in SPLIT_N},
           "answer_allowance": ANSWER_ALLOWANCE,
           "note": "matched in thinking tokens; the forced answer is overhead on top"},
    forcing={"stem": FORCE_STEM, "answer_allowance": ANSWER_ALLOWANCE},
    items={"source": "MATH-500 (ch03 copy)",
           "dataset": DATASET, "n": len(items), "item_set_sha256": ITEM_SET_HASH,
           "seed": SEED, "same_as_page_6": SAME_ITEMS_AS_PAGE_6},
    vote={"tally_key": "reasoning_from_scratch.ch03.extract_final_candidate"
                       " (fallback='number_only')",
          "equivalence": "reasoning_from_scratch.ch03.grade_answer, both directions",
          "tie_rule": "a tie returns no answer and grades wrong (ch04 self_consistency_vote)"},
    grading={"extract": "reasoning_from_scratch.ch03.extract_final_candidate",
             "grade": "reasoning_from_scratch.ch03.grade_answer"},
    prompt_template=render_prompt("{question}"),
    versions={"torch": torch.__version__,
              "transformers": __import__("transformers").__version__,
              "device": str(device) if device is not None else None},
    cost={"planned_tokens": TOTAL_TOKENS, "batched_tok_per_s": BATCH_TOK_S,
          "priced_from": "page 6's measured spend per cap, not the caps"},
    integrity=INTEGRITY,
)

gt_of = {pid: rs[0]["ground_truth"] for pid, rs in PRIMARY_SAMPLES[POOL_CONFIG].items()}
harness.write_json(RESULTS_PATH, dict(
    run_id=RUN_ID, primary_model=SWEPT[0], claims=CLAIMS,
    pile=dict(config_id=POOL_CONFIG, n_chains=POOL_N, temperature=T_BEST, chain_cap=CHAIN_CAP),
    # one tally per problem, each answer class graded
    pile_rows={pid: dict(row, ranked=[{"answer": a, "votes": c, "correct": bool(grade_answer(a, gt_of[pid]))}
                                     for a, c in row["ranked"]])
               for pid, row in POOL_ROWS.items()},
    specimen=dict(specimen_id=SPECIMEN_ID, specimen_is_fallback=SPECIMEN_IS_FALLBACK,
                  n_candidates=len(candidates), shape=SHAPE),
    pile_curves=[{"temperature": t, "points": pts} for t, pts in PILE.items()],
    split=SPLIT, splitting_pays=SPLIT_PAYS, best_split_n=peak["n"],
    # `n_items` here is the diversity family's smaller item set, not the pile's.
    diversity=[dict(temperature=t, n_chains=DIVERSITY_N, **a) for t, a in sorted(D_AGG.items())],
    # `vs_bare_cot` is the contrast the page quotes.
    gain=dict(vs_one_sample=dict(pts=GAIN_PTS, p=p_value, fixed=fixed, broken=broken,
                                 baseline=A_POOL["single_chain_accuracy"]),
              vs_bare_cot=BARE, vote=A_POOL["plurality_accuracy"], n_chains=POOL_N),
    handoff=HANDOFF,
))

for path in (RESULTS_PATH, MANIFEST_PATH, TRACES_PATH):
    print(f"{path.name:<15}{harness.kb(path):>10,.1f} KB")
print()
for k, v in CLAIMS.items():
    print(f"   {'ok ' if v else 'NO '} {k}")